In [ ]:
# Import Data manipulation libraries and visualization libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Imprt filter warnings to ignore warnings
import warnings
warnings.filterwarnings("ignore")

# Import OrderedDict
from collections import OrderedDict

# Import Loggings 
import logging
logging.basicConfig(level=logging.INFO, 
                    format='%(asctime)s - %(levelname)s - %(message)s',
                    filename = 'Phissing.log',
                    filemode = 'w',
                    force = True)

# Import Sklearn libraries for machine learning
from sklearn.model_selection import train_test_split,StratifiedKFold
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from sklearn.metrics import classification_report,accuracy_score,confusion_matrix

# Importing models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, BaggingClassifier
from sklearn.ensemble import AdaBoostClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import xgboost
from xgboost import XGBClassifier

# Setting the Theme style so that it shows no truncations
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [2]:
# Data Ingestion (Importing the dataset)

df = pd.read_csv(r'C:\Phissing_Prediction_Model\data\raw\Phissing_Dataset.csv')
df.head()

,url,length_url,length_hostname,ip,nb_dots,nb_hyphens,nb_at,nb_qm,nb_and,nb_or,nb_eq,nb_underscore,nb_tilde,nb_percent,nb_slash,nb_star,nb_colon,nb_comma,nb_semicolumn,nb_dollar,nb_space,nb_www,nb_com,nb_dslash,http_in_path,https_token,ratio_digits_url,ratio_digits_host,punycode,port,tld_in_path,tld_in_subdomain,abnormal_subdomain,nb_subdomains,prefix_suffix,random_domain,shortening_service,path_extension,nb_redirection,nb_external_redirection,length_words_raw,char_repeat,shortest_words_raw,shortest_word_host,shortest_word_path,longest_words_raw,longest_word_host,longest_word_path,avg_words_raw,avg_word_host,avg_word_path,phish_hints,domain_in_brand,brand_in_subdomain,brand_in_path,suspecious_tld,statistical_report,nb_hyperlinks,ratio_intHyperlinks,ratio_extHyperlinks,ratio_nullHyperlinks,nb_extCSS,ratio_intRedirection,ratio_extRedirection,ratio_intErrors,ratio_extErrors,login_form,external_favicon,links_in_tags,submit_email,ratio_intMedia,ratio_extMedia,sfh,iframe,popup_window,safe_anchor,onmouseover,right_clic,empty_title,domain_in_title,domain_with_copyright,whois_registered_domain,domain_registration_length,domain_age,web_traffic,dns_record,google_index,page_rank,status
0,http://www.crestonwood.com/router.php,37,19,0,3,0,0,0,0,0,0,0,0,0,3,0,1,0,0,0,0,1,0,0,0,1,0.000000,0.0,0,0,0,0,0,3,0,0,0,0,0,0,4,4,3,3,3,11,11,6,5.750000,7.0,4.500000,0,0,0,0,0,0,17,0.529412,0.470588,0,0,0,0.875000,0,0.500000,0,0,80.000000,0,100.000000,0.000000,0,0,0,0.0,0,0,0,0,1,0,45,-1,0,1,1,4,legitimate
1,http://shadetreetechnology.com/V4/validation/a...,77,23,1,1,0,0,0,0,0,0,0,0,0,5,0,1,0,0,0,0,0,0,0,0,1,0.220779,0.0,0,0,0,0,0,1,0,0,0,0,1,0,4,4,2,19,2,32,19,32,15.750000,19.0,14.666667,0,0,0,0,0,0,30,0.966667,0.033333,0,0,0,0.000000,0,0.000000,0,0,100.000000,0,80.000000,20.000000,0,0,0,100.0,0,0,0,1,0,0,77,5767,0,0,1,2,phishing
2,https://support-appleld.com.secureupdate.duila...,126,50,1,4,1,0,1,2,0,3,2,0,0,5,0,1,0,0,0,0,0,1,0,0,0,0.150794,0.0,0,0,0,1,0,3,1,0,0,0,1,0,12,2,2,3,2,17,13,17,8.250000,8.4,8.142857,0,0,0,0,0,0,4,1.000000,0.000000,0,0,0,0.000000,0,0.000000,0,0,100.000000,0,0.000000,0.000000,0,0,0,100.0,0,0,0,1,0,0,14,4004,5828815,0,1,0,phishing
3,http://rgipt.ac.in,18,11,0,2,0,0,0,0,0,0,0,0,0,2,0,1,0,0,0,0,0,0,0,0,1,0.000000,0.0,0,0,0,0,0,2,0,0,0,0,1,0,1,0,5,5,0,5,5,0,5.000000,5.0,0.000000,0,0,0,0,0,0,149,0.973154,0.026846,0,0,0,0.250000,0,0.250000,0,0,100.000000,0,96.428571,3.571429,0,0,0,62.5,0,0,0,1,0,0,62,-1,107721,0,0,3,legitimate
4,http://www.iracing.com/tracks/gateway-motorspo...,55,15,0,2,2,0,0,0,0,0,0,0,0,5,0,1,0,0,0,0,1,0,0,0,1,0.000000,0.0,0,0,0,0,0,2,0,0,0,0,1,0,6,3,3,3,4,11,7,11,6.333333,5.0,7.000000,0,0,0,0,0,0,102,0.470588,0.529412,0,0,0,0.537037,0,0.018519,1,0,76.470588,0,0.000000,100.000000,0,0,0,0.0,0,0,0,0,1,0,224,8175,8725,0,0,6,legitimate


In [3]:
# Checking the information of the dataset
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11430 entries, 0 to 11429
Data columns (total 89 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   url                         11430 non-null  object 
 1   length_url                  11430 non-null  int64  
 2   length_hostname             11430 non-null  int64  
 3   ip                          11430 non-null  int64  
 4   nb_dots                     11430 non-null  int64  
 5   nb_hyphens                  11430 non-null  int64  
 6   nb_at                       11430 non-null  int64  
 7   nb_qm                       11430 non-null  int64  
 8   nb_and                      11430 non-null  int64  
 9   nb_or                       11430 non-null  int64  
 10  nb_eq                       11430 non-null  int64  
 11  nb_underscore               11430 non-null  int64  
 12  nb_tilde                    11430 non-null  int64  
 13  nb_percent                  114

In [4]:
 # Checking the number of missing values in each column
df.isnull().sum()

url                           0
length_url                    0
length_hostname               0
ip                            0
nb_dots                       0
nb_hyphens                    0
nb_at                         0
nb_qm                         0
nb_and                        0
nb_or                         0
nb_eq                         0
nb_underscore                 0
nb_tilde                      0
nb_percent                    0
nb_slash                      0
nb_star                       0
nb_colon                      0
nb_comma                      0
nb_semicolumn                 0
nb_dollar                     0
nb_space                      0
nb_www                        0
nb_com                        0
nb_dslash                     0
http_in_path                  0
https_token                   0
ratio_digits_url              0
ratio_digits_host             0
punycode                      0
port                          0
tld_in_path                   0
tld_in_s

In [5]:
# Shape of the Dataset
df.shape

(11430, 89)

In [6]:
numerical_cols = df.select_dtypes(include='number')
Q1 = numerical_cols.quantile(0.25)
Q3 = numerical_cols.quantile(0.75)
IQR = Q3 - Q1
print(IQR)

length_url                        38.000000
length_hostname                    9.000000
ip                                 0.000000
nb_dots                            1.000000
nb_hyphens                         1.000000
nb_at                              0.000000
nb_qm                              0.000000
nb_and                             0.000000
nb_or                              0.000000
nb_eq                              0.000000
nb_underscore                      0.000000
nb_tilde                           0.000000
nb_percent                         0.000000
nb_slash                           2.000000
nb_star                            0.000000
nb_colon                           0.000000
nb_comma                           0.000000
nb_semicolumn                      0.000000
nb_dollar                          0.000000
nb_space                           0.000000
nb_www                             1.000000
nb_com                             0.000000
nb_dslash                       

In [7]:
cols_to_drop = []
for col in df.select_dtypes(include='number').columns:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    if IQR == 0:
        cols_to_drop.append(col)
print("Columns to drop:", cols_to_drop)
df = df.drop(columns=cols_to_drop)

Columns to drop: ['ip', 'nb_at', 'nb_qm', 'nb_and', 'nb_or', 'nb_eq', 'nb_underscore', 'nb_tilde', 'nb_percent', 'nb_star', 'nb_colon', 'nb_comma', 'nb_semicolumn', 'nb_dollar', 'nb_space', 'nb_com', 'nb_dslash', 'http_in_path', 'ratio_digits_host', 'punycode', 'port', 'tld_in_path', 'tld_in_subdomain', 'abnormal_subdomain', 'prefix_suffix', 'random_domain', 'shortening_service', 'path_extension', 'nb_external_redirection', 'phish_hints', 'domain_in_brand', 'brand_in_subdomain', 'brand_in_path', 'suspecious_tld', 'statistical_report', 'ratio_nullHyperlinks', 'ratio_intRedirection', 'ratio_intErrors', 'login_form', 'submit_email', 'sfh', 'iframe', 'popup_window', 'onmouseover', 'right_clic', 'empty_title', 'domain_in_title', 'whois_registered_domain', 'dns_record']


In [8]:
print("Remaining Columns:", df.columns)

Remaining Columns: Index(['url', 'length_url', 'length_hostname', 'nb_dots', 'nb_hyphens',
       'nb_slash', 'nb_www', 'https_token', 'ratio_digits_url',
       'nb_subdomains', 'nb_redirection', 'length_words_raw', 'char_repeat',
       'shortest_words_raw', 'shortest_word_host', 'shortest_word_path',
       'longest_words_raw', 'longest_word_host', 'longest_word_path',
       'avg_words_raw', 'avg_word_host', 'avg_word_path', 'nb_hyperlinks',
       'ratio_intHyperlinks', 'ratio_extHyperlinks', 'nb_extCSS',
       'ratio_extRedirection', 'ratio_extErrors', 'external_favicon',
       'links_in_tags', 'ratio_intMedia', 'ratio_extMedia', 'safe_anchor',
       'domain_with_copyright', 'domain_registration_length', 'domain_age',
       'web_traffic', 'google_index', 'page_rank', 'status'],
      dtype='object')


In [9]:
target = 'status'
le = LabelEncoder()
df[target] = le.fit_transform(df[target])
X = df.drop(columns = [target])
y = df[target]

In [10]:
X_train, X_test, y_train, y_test = train_test_split(X, 
                                                    y, 
                                                    test_size=0.3, 
                                                    random_state=42, 
                                                    stratify=y)
# stratify = y is used for balancing the classes for the training and testing datasets

In [11]:
numerical_cols = X_train.select_dtypes(exclude='object').columns
categorical_cols = X_train.select_dtypes(include='object').columns

# 2. Create preprocessing pipelines
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', MinMaxScaler())
])

cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# 3. Create preprocessor
preprocessor = ColumnTransformer([
    ('num', num_pipeline, numerical_cols),
    ('cat', cat_pipeline, categorical_cols)
])

# 4. Create pipeline with SMOTE then PCA
final_pipeline = ImbPipeline([
    ('preprocessor', preprocessor),
    ('smote', SMOTE(random_state=42)),
    ('pca', PCA(n_components=0.95, random_state=42))
])

# 5. For TRAINING data: Apply preprocessing, SMOTE, then PCA
print("Processing training data...")
# First apply preprocessing
X_train_preprocessed = final_pipeline.named_steps['preprocessor'].fit_transform(X_train)

# Then apply SMOTE (this creates synthetic samples)
X_train_smoted, y_train_resampled = final_pipeline.named_steps['smote'].fit_resample(
    X_train_preprocessed, y_train
)
print(f"Original training size: {len(X_train)} → After SMOTE: {len(X_train_smoted)}")

# Finally apply PCA
pca = final_pipeline.named_steps['pca']
X_train_resampled = pca.fit_transform(X_train_smoted)
print(f"After PCA: {X_train_resampled.shape[1]} components")

# 6. For TEST data: Apply preprocessing and PCA only (NO SMOTE!)
print("\nProcessing test data...")
X_test_preprocessed = final_pipeline.named_steps['preprocessor'].transform(X_test)
X_test_transformed = pca.transform(X_test_preprocessed)  # Use same PCA fitted on training


Processing training data...


Original training size: 8001 → After SMOTE: 8002
After PCA: 6653 components

Processing test data...


In [29]:
from flaml import AutoML

# Initialize AutoML
automl = AutoML()

# Settings for FLAML
settings = {
    "time_budget": 300,  # seconds; adjust for bigger datasets
    "metric": "accuracy",  # you can also use 'f1', 'roc_auc'
    "task": "classification",
    "estimator_list": ["rf", "extra_tree", "xgboost", "lrl2"],
    "log_file_name": "flaml.log",
    "seed": 42
}

# Fit FLAML on training data
automl.fit(X_train=X_train_resampled, y_train=y_train_resampled, **settings)

print("FLAML Training Completed")

# Safely print the best estimator
best_model = getattr(automl, "best_estimator", None)
if best_model is not None:
    print("Best Model:", best_model)
else:
    print("FLAML did not find a model. Try increasing time_budget.")

# Make predictions on test data
y_pred = automl.predict(X_test_transformed)

# Evaluate
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

print("Accuracy:", accuracy_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))

[flaml.automl.logger: 03-07 02:23:54] {2375} INFO - task = classification
[flaml.automl.logger: 03-07 02:23:54] {2386} INFO - Evaluation method: holdout
[flaml.automl.logger: 03-07 02:23:55] {2489} INFO - Minimizing error metric: 1-accuracy
[flaml.automl.logger: 03-07 02:23:55] {2606} INFO - List of ML learners in AutoML Run: ['rf', 'extra_tree', 'xgboost', 'lrl2']
[flaml.automl.logger: 03-07 02:23:55] {2911} INFO - iteration 0, current learner rf
[flaml.automl.logger: 03-07 02:23:55] {3046} INFO - Estimated sufficient time budget=4321s. Estimated necessary time budget=7s.
[flaml.automl.logger: 03-07 02:23:55] {3097} INFO -  at 1.4s,	estimator rf's best error=3.3583e-01,	best estimator rf's best error=3.3583e-01
[flaml.automl.logger: 03-07 02:23:55] {2911} INFO - iteration 1, current learner xgboost
[flaml.automl.logger: 03-07 02:24:03] {3097} INFO -  at 8.7s,	estimator xgboost's best error=1.4732e-01,	best estimator xgboost's best error=1.4732e-01
[flaml.automl.logger: 03-07 02:24:03]

In [24]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Predictions
y_train_pred = automl.predict(X_train)
y_test_pred = automl.predict(X_test)

# Train metrics
print("Train Accuracy:", accuracy_score(y_train, y_train_pred))
print("Train Precision:", precision_score(y_train, y_train_pred))
print("Train Recall:", recall_score(y_train, y_train_pred))
print("Train F1:", f1_score(y_train, y_train_pred))

# Test metrics
print("Test Accuracy:", accuracy_score(y_test, y_test_pred))
print("Test Precision:", precision_score(y_test, y_test_pred))
print("Test Recall:", recall_score(y_test, y_test_pred))
print("Test F1:", f1_score(y_test, y_test_pred))

Train Accuracy: 1.0
Train Precision: 1.0
Train Recall: 1.0
Train F1: 1.0
Test Accuracy: 0.9655876348789735
Test Precision: 0.9618055555555556
Test Recall: 0.969661610268378
Test F1: 0.9657176060429983


In [25]:
from sklearn.metrics import roc_auc_score

roc_auc_score(y_test, automl.predict_proba(X_test)[:,1])

0.9928916724215942

In [27]:
feature_names = X_train.columns
print("Feature Names:", feature_names)

Feature Names: Index(['url', 'length_url', 'length_hostname', 'nb_dots', 'nb_hyphens',
       'nb_slash', 'nb_www', 'https_token', 'ratio_digits_url',
       'nb_subdomains', 'nb_redirection', 'length_words_raw', 'char_repeat',
       'shortest_words_raw', 'shortest_word_host', 'shortest_word_path',
       'longest_words_raw', 'longest_word_host', 'longest_word_path',
       'avg_words_raw', 'avg_word_host', 'avg_word_path', 'nb_hyperlinks',
       'ratio_intHyperlinks', 'ratio_extHyperlinks', 'nb_extCSS',
       'ratio_extRedirection', 'ratio_extErrors', 'external_favicon',
       'links_in_tags', 'ratio_intMedia', 'ratio_extMedia', 'safe_anchor',
       'domain_with_copyright', 'domain_registration_length', 'domain_age',
       'web_traffic', 'google_index', 'page_rank'],
      dtype='object')
